# CTC Detection — Geneformer Fine-tuning
Fine-tunes Geneformer for circulating tumor cell (CTC) detection in scRNA-seq data.
Targets EpCAM-low CTCs missed by CellSearch.

**Runtime:** GPU (T4 or better). Runtime > Change runtime type > T4 GPU

## 1. Check GPU

In [ ]:
!git clone https://github.com/gabufle/ctc-detect.git
%cd ctc-detect
!pip install transformers==4.41.0 peft==0.11.0 accelerate datasets scanpy anndata pyensembl mygene -q
!pyensembl install --release 109 --species homo_sapiens

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# copy h5ad from drive to working directory
!cp /content/drive/MyDrive/ctc_merged_processed.h5ad data/processed/

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 2. Install dependencies

In [ ]:
#%%capture
!pip install scanpy anndata peft accelerate datasets scikit-learn umap-learn matplotlib seaborn
!git clone https://huggingface.co/ctheodoris/Geneformer
!pip install --no-deps ./Geneformer
!pip install loompy tdigest -q
!pip install mygene -q

## 3. Mount Google Drive and load data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('data/processed', exist_ok=True)
os.makedirs('results/checkpoints/best_model', exist_ok=True)
os.makedirs('results/figures', exist_ok=True)

# Copy h5ad from Drive
!cp /content/drive/MyDrive/ctc_merged_processed.h5ad data/processed/
print('h5ad loaded')

## 4. Load and inspect data

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd

adata = sc.read_h5ad('/content/drive/MyDrive/CTC - testing/ctc_merged_processed.h5ad')
print('Cells:', adata.n_obs)
print('Genes:', adata.n_vars)
print('Columns:', adata.obs.columns.tolist())
print()
print('CTC breakdown:')
print(adata.obs['is_ctc'].value_counts())
print()
print('EpCAM status breakdown:')
print(adata.obs['epcam_status'].value_counts())

## 5. Tokenize for Geneformer

In [ ]:
# Geneformer needs Ensembl IDs - convert gene symbols
# First check what format our genes are in
print('First 10 gene names:', adata.var_names[:10].tolist())
print('Gene name format looks like Ensembl?', adata.var_names[0].startswith('ENSG'))

In [ ]:
from pyensembl import EnsemblRelease
genome = EnsemblRelease(109)

def symbol_to_ensembl(symbol):
    try:
        genes = genome.genes_by_name(symbol)
        if genes:
            return genes[0].gene_id
        return None
    except:
        return None

print('Converting gene symbols to Ensembl IDs...')
adata.var['ensembl_id'] = [symbol_to_ensembl(g) for g in adata.var_names]

# Report conversion rate
n_converted = adata.var['ensembl_id'].notna().sum()
print(f'Converted: {n_converted}/{adata.n_vars} genes ({100*n_converted/adata.n_vars:.1f}%)')

# Filter to converted genes only
adata = adata[:, adata.var['ensembl_id'].notna()].copy()
adata.var_names = adata.var['ensembl_id'].values
print(f'Genes after filtering: {adata.n_vars}')
print('First 5 Ensembl IDs:', adata.var_names[:5].tolist())

In [ ]:
import os
for root, dirs, files in os.walk('./Geneformer'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
import pickle
import numpy as np
from scipy.sparse import issparse

# Load Geneformer V2 vocabulary files
with open('./Geneformer/geneformer/gene_median_dictionary_gc104M.pkl', 'rb') as f:
    gene_median_dict = pickle.load(f)

with open('./Geneformer/geneformer/token_dictionary_gc104M.pkl', 'rb') as f:
    token_dict = pickle.load(f)

print(f'Vocab size: {len(token_dict)} tokens')
print(f'Genes in median dict: {len(gene_median_dict)}')

def tokenize_cell(expression_vector, gene_names):
    if issparse(expression_vector):
        expression_vector = expression_vector.toarray().flatten()
    else:
        expression_vector = np.array(expression_vector).flatten()

    expressed_mask = expression_vector > 0
    expressed_genes = gene_names[expressed_mask]
    expressed_values = expression_vector[expressed_mask]

    normalized = []
    for gene, val in zip(expressed_genes, expressed_values):
        median = gene_median_dict.get(gene, None)
        if median and median > 0:
            normalized.append((gene, val / median))

    normalized.sort(key=lambda x: x[1], reverse=True)
    token_ids = [token_dict[g] for g, _ in normalized if g in token_dict]
    return token_ids

print('Tokenizing cells...')
gene_names = np.array(adata.var_names)
all_input_ids = []
all_lengths = []

for i in range(adata.n_obs):
    if i % 1000 == 0:
        print(f'  {i}/{adata.n_obs}')
    expr = adata.X[i]
    token_ids = tokenize_cell(expr, gene_names)
    all_input_ids.append(token_ids)
    all_lengths.append(len(token_ids))

print(f'Done. Mean length: {np.mean(all_lengths):.0f} | Max: {max(all_lengths)} | Min: {min(all_lengths)}')

## 6. Split, pad and prepare datasets

In [ ]:
from datasets import Dataset
import pandas as pd

# Build dataset from tokenized cells
dataset_dict = {
    'input_ids': all_input_ids,
    'length': all_lengths,
    'label': [int(x) for x in adata.obs['is_ctc'].values],
    'is_ctc': [int(x) for x in adata.obs['is_ctc'].values],
    'epcam_status': [str(x) for x in adata.obs['epcam_status'].values],
}

full_ds = Dataset.from_dict(dataset_dict)
print(full_ds)
print(f'CTCs: {sum(full_ds["label"])} | Normal: {len(full_ds) - sum(full_ds["label"])}')

In [ ]:
from datasets import DatasetDict

# Stratified split
ctc_idx    = [i for i, x in enumerate(full_ds['label']) if x == 1]
normal_idx = [i for i, x in enumerate(full_ds['label']) if x == 0]
np.random.seed(42)
np.random.shuffle(ctc_idx)
np.random.shuffle(normal_idx)

def split_indices(idx, train=0.7, val=0.15):
    n = len(idx)
    return idx[:int(n*train)], idx[int(n*train):int(n*(train+val))], idx[int(n*(train+val)):]

ctc_tr, ctc_va, ctc_te       = split_indices(ctc_idx)
norm_tr, norm_va, norm_te    = split_indices(normal_idx)

train_idx = sorted(ctc_tr + norm_tr)
val_idx   = sorted(ctc_va + norm_va)
test_idx  = sorted(ctc_te + norm_te)

train_ds = full_ds.select(train_idx)
val_ds   = full_ds.select(val_idx)
test_ds  = full_ds.select(test_idx)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
print(f'Train CTCs: {sum(train_ds["label"])} | Normal: {len(train_ds) - sum(train_ds["label"])}')

In [ ]:
# Pad and truncate to 2048 (Geneformer pretraining max)
MAX_LENGTH = 2048

def pad_and_truncate(example):
    ids = example['input_ids'][:MAX_LENGTH]
    length = len(ids)
    example['input_ids'] = ids + [0] * (MAX_LENGTH - length)
    example['attention_mask'] = [1] * length + [0] * (MAX_LENGTH - length)
    example['length'] = length
    return example

train_ds = train_ds.map(pad_and_truncate, num_proc=2)
val_ds   = val_ds.map(pad_and_truncate, num_proc=2)
test_ds  = test_ds.map(pad_and_truncate, num_proc=2)

# Set torch format
cols = ['input_ids', 'attention_mask', 'label']
train_ds.set_format(type='torch', columns=cols)
val_ds.set_format(type='torch', columns=cols)
test_ds.set_format(type='torch', columns=cols)

print('Sequence length check:', len(train_ds[0]['input_ids']))
print('Ready for training')

## 7. Load model and apply LoRA

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print(os.path.exists('/content/drive/MyDrive/ctc_tokenized_train'))
print(os.path.exists('/content/drive/MyDrive/ctc_tokenized_val'))
print(os.path.exists('/content/drive/MyDrive/ctc_tokenized_test'))

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = './Geneformer/Geneformer-V1-10M'

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True,
)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=['query', 'value'],
    bias='none',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
!mkdir -p /content/drive/MyDrive/ctc_detect_data
full_ds.save_to_disk('/content/drive/MyDrive/ctc_detect_data/full')
train_ds.save_to_disk('/content/drive/MyDrive/ctc_detect_data/train')
val_ds.save_to_disk('/content/drive/MyDrive/ctc_detect_data/val')
test_ds.save_to_disk('/content/drive/MyDrive/ctc_detect_data/test')
print('backed up to drive')

## 8. Train

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback, default_data_collator
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

# Class weights
labels_arr = np.array(train_ds['label'])
n_normal = (labels_arr == 0).sum()
n_ctc    = (labels_arr == 1).sum()
w = torch.tensor([1.0/n_normal, 1.0/n_ctc], dtype=torch.float32)
w = w / w.sum()
print(f'Class weights — Normal: {w[0]:.4f} | CTC: {w[1]:.4f}')
print(f'Class balance — Normal: {n_normal} | CTC: {n_ctc}')

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = nn.CrossEntropyLoss(weight=w.to(outputs.logits.device))(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    preds = (probs >= 0.5).astype(int)
    if len(np.unique(labels)) < 2:
        return {'auroc': 0.0, 'auprc': 0.0, 'sensitivity': 0.0, 'specificity': 0.0}
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0,1]).ravel()
    return {
        'auroc':       roc_auc_score(labels, probs),
        'auprc':       average_precision_score(labels, probs),
        'sensitivity': tp / max(tp+fn, 1),
        'specificity': tn / max(tn+fp, 1),
    }

training_args = TrainingArguments(
    output_dir='results/checkpoints/best_model',
    num_train_epochs=20,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=1e-4,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='auprc',
    greater_is_better=True,
    logging_steps=10,
    fp16=True,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    report_to='none',
    max_grad_norm=1.0,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    data_collator=default_data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print('Starting training...')
train_result = trainer.train()
print('Training complete')

## 9. Evaluate on test set

In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay
import matplotlib.pyplot as plt

test_result  = trainer.predict(test_ds)
probs        = torch.softmax(torch.tensor(test_result.predictions), dim=-1).numpy()[:, 1]
labels       = test_result.label_ids
preds        = (probs >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0,1]).ravel()
auroc       = roc_auc_score(labels, probs)
auprc       = average_precision_score(labels, probs)
sensitivity = tp / max(tp+fn, 1)
specificity = tn / max(tn+fp, 1)

print('=== TEST SET RESULTS ===')
print(f'AUROC:       {auroc:.4f}')
print(f'AUPRC:       {auprc:.4f}')
print(f'Sensitivity: {sensitivity:.4f}')
print(f'Specificity: {specificity:.4f}')

if sensitivity < 0.7:
    print('WARNING: Sensitivity below 0.7 — model missing too many CTCs')

# ROC and PR curves
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
RocCurveDisplay.from_predictions(labels, probs, ax=axes[0], name='Geneformer-CTC')
PrecisionRecallDisplay.from_predictions(labels, probs, ax=axes[1], name='Geneformer-CTC')
axes[0].set_title('ROC Curve')
axes[1].set_title('Precision-Recall Curve')
plt.tight_layout()
plt.savefig('results/figures/roc_pr_curves.png', dpi=150)
plt.show()
print('Saved to results/figures/roc_pr_curves.png')

## 10. EpCAM subgroup analysis

In [ ]:
# Get epcam status for test cells
test_epcam = [str(full_ds[i]['epcam_status']) for i in test_idx]

results_df = pd.DataFrame({
    'label':        labels,
    'prob':         probs,
    'pred':         preds,
    'epcam_status': test_epcam,
})

print('=== EPCAM SUBGROUP ANALYSIS ===')
for status in results_df['epcam_status'].unique():
    subset = results_df[results_df['epcam_status'] == status]
    ctc_subset = subset[subset['label'] == 1]
    if len(ctc_subset) == 0:
        continue
    detected = (ctc_subset['pred'] == 1).sum()
    sensitivity_sub = detected / len(ctc_subset)
    print(f'\nEpCAM status: {status}')
    print(f'  CTC cells:   {len(ctc_subset)}')
    print(f'  Detected:    {detected}')
    print(f'  Sensitivity: {sensitivity_sub:.4f}')

## 11. UMAP visualization

In [ ]:
import scanpy as sc
import seaborn as sns

# Build anndata from test set for UMAP
test_input_ids = np.array(test_ds['input_ids'])

adata_test = sc.AnnData(
    X=test_input_ids.astype(np.float32),
)
adata_test.obs['true_label']   = ['CTC' if l == 1 else 'Normal' for l in labels]
adata_test.obs['ctc_prob']     = probs
adata_test.obs['predicted']    = ['CTC' if p == 1 else 'Normal' for p in preds]
adata_test.obs['uncertain']    = ((probs >= 0.4) & (probs <= 0.6)).astype(str)
adata_test.obs['epcam_status'] = test_epcam

sc.pp.pca(adata_test, n_comps=50)
sc.pp.neighbors(adata_test)
sc.tl.umap(adata_test)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
sc.pl.umap(adata_test, color='true_label',   ax=axes[0,0], show=False, title='Ground Truth')
sc.pl.umap(adata_test, color='ctc_prob',     ax=axes[0,1], show=False, title='CTC Probability', cmap='Reds')
sc.pl.umap(adata_test, color='predicted',    ax=axes[1,0], show=False, title='Predicted Label')
sc.pl.umap(adata_test, color='epcam_status', ax=axes[1,1], show=False, title='EpCAM Status')
plt.tight_layout()
plt.savefig('results/figures/umap_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to results/figures/umap_predictions.png')

## 12. Save checkpoint to Google Drive

In [ ]:
import shutil

trainer.save_model('results/checkpoints/best_model')

# Copy checkpoint and results to Drive
!cp -r results/checkpoints/best_model /content/drive/MyDrive/ctc_detect_checkpoint
!cp -r results/figures /content/drive/MyDrive/ctc_detect_figures

print('Checkpoint saved to Google Drive: ctc_detect_checkpoint/')
print('Figures saved to Google Drive: ctc_detect_figures/')
print('Done!')

In [ ]:
trainer.save_model('results/checkpoints/best_model')
!cp -r results/checkpoints/best_model /content/drive/MyDrive/ctc_detect_checkpoint
!cp -r results/figures /content/drive/MyDrive/ctc_detect_figures
print('saved')

In [ ]:
# download figures and results locally first
from google.colab import files

# zip results folder
!zip -r results.zip results/figures/ results/metrics.json
files.download('results.zip')